In [51]:
import sys
sys.path.insert(1, '../scripts/') # comment out in python script
from load_environmental_variables import *
from utils import *

In [537]:
rxnAbbreviation

['Reaction abbreviation',
 'SEC61C',
 'TRAP',
 'ARF1_gdp_binding',
 'ARF1_activation',
 'ARF1_gdp_degradation',
 'SAR1B_gdp_binding',
 'SAR1B_activation',
 'SAR1B_gdp_degradation',
 'RAB1A_gdp_binding',
 'RAB6B_gdp_binding',
 'RAB8B_gdp_binding',
 'RAB1A_gdp_degradation',
 'RAB6B_gdp_degradation',
 'RAB8B_gdp_degradation',
 'RAB1A_activation',
 'RAB6B_activation',
 'RAB8B_activation',
 'BiP_NEF',
 'BiP_ATPase',
 'BiP_atp_formation',
 'BiP_adp_degradation',
 'GOLGI_TO_ENDOSOME_3',
 'COPI_recycling',
 'COPII_recycling',
 'NSF-dissociation',
 'retro_TRANSLOC_1',
 'Proteasome_complex',
 'ASNA1_atp',
 'ASNA1_adp_degradation',
 'ASNA1_formation',
 'CALR_Ca2',
 'CALR_Ca2_degradation',
 'GTHOX_CtoR_transport',
 'PDI_reoxidation_GSSG',
 'PDI_reoxidation_ERO1LB',
 'PDI_reoxidation_H2O2',
 'ERO1LB_reoxidation_1',
 'ERO1A_reoxidation_1',
 'PDI_reoxidation_ERO1A',
 'EXOCYST_COMPLEX',
 'EXOCYST_degradation',
 'SECRETION_3',
 'SRP',
 'SRPR',
 'SPC',
 'SPC_degradation',
 'co_TRANSLOC_7',
 'TRAP_degrad

In [538]:
rxnConditions

['Conditions',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '(Localization=[d])OR(Localization=[x])OR(Localization=[l])',
 '(Localization=Endosome)OR(Localization=Peroxisome)',
 '(Location = [rm])OR(Location = [r])',
 '(Location = [rm])OR(Location = [r])',
 '(Location = [rm])OR(Location = [r])',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 'NG>0',
 'NG>0',
 'NG>0',
 'NG>0',
 'NG>0',
 'NG>0',
 'NG>0',
 'NG>0',
 'NG>0',
 'NG>0',
 'NG>0',
 'NG>0',
 '',
 'NG>0',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 '',
 'OG>0',
 'OG>0',
 'OG>0',
 '(NG>0)AND(protein=EPO)',
 '(NG>0)AND(protein=EPO)',
 '(NG>0)AND(protein=EPO)',
 'GPI=1',
 'GPI=1',
 '(SP=1)AND(len(XXX)<=160)',
 '(SP=1)AND(len(XXX)<=160)',
 '(SP=1)AND(len(XXX)<=160)AND(Localization=Secret

In [7]:
f_path = root_path + 'MammalianSecretoryRecon/JUPYTER_NOTEBOOKS/RECON2s_python/'
# Read files with appropriate template RECON2s reactions and remove "\n"
f = open(f_path + "rxnFormula_HUMAN.txt")
rxnFormula = f.read().splitlines()
f.close()

f = open(f_path + "rxnAbbreviation_HUMAN.txt")
rxnAbbreviation = f.read().splitlines()
f.close()

f = open(f_path + "rxnPathway_HUMAN.txt")
rxnPathway = f.read().splitlines()
f.close()

f = open(f_path + "rxnConditions_HUMAN.txt")
rxnConditions = f.read().splitlines()
f.close()

f = open(f_path + "rxnGPRs_HUMAN.txt")
rxnGPR = f.read().splitlines()
f.close()

# Load PSIM matrix
f = open(f_path + 'PSIM_HUMAN.tab','r')
PSIM=f.read().splitlines()
f.close()

#Extract the Uniprot IDs
PSIM_entries = []
for line in PSIM:
    PSIM_entries.append(line.split('\t')[0])

	# Define Basic Functions

#Define a functions that counts of amino acids
def count_AAs(sequence):
    AAcounts = []    
    AAs = ['G', 'A', 'V', 'L', 'I', 'M', 'W', 'F', 'P', 'S', 'T', 'C', 'Y', 'N', 'Q', 'E', 'D', 'K', 'R', 'H']
    for aa in AAs:
        AAcounts.append(sequence.count(aa))
    return AAcounts

#Define a function that substitutes question marks for AA counts in a formula (string)
#Useful for translation and protein degradation pathway

def substitute_AAs_count(formula,AAcounts):
    template = "? gly[c] + ? ala_L[c] + ? val_L[c] + ? leu_L[c] + ? ile_L[c] + ? met_L[c] + ? trp_L[c] + ? phe_L[c] + ? pro_L[c] + ? ser_L[c] + ? thr_L[c] + ? cys_L[c] + ? tyr_L[c] + ? asn_L[c] + ? gln_L[c] + ? glu_L[c] + ? asp_L[c] + ? lys_L[c] + ? arg_L[c] + ? his_L[c]"
    new = ''
    index = 0
    for character in template:
        if character == "?":
            new = new + str(AAcounts[index])
            index = index + 1
        else:
            new = new + character
    newFormula = formula.replace(template,new)
    return newFormula

# Function that generates the translation reaction of a protein given its UniProt ID
def translate_protein(entryID):
    #Obtain protein sequence and length
    PSI_row = PSIM[PSIM_entries.index(str(entryID))] 
    PSI_row = PSI_row.split('\t')
    sequence = PSI_row[11]
    AAcounts = count_AAs(sequence)
    N = len(sequence) #Number to replace atp's, gtp's, ppi's, pi's, h2o's, amp's, and gpd's
    templateFormula = "? h2o[c] + ? atp[c] + ? gtp[c] + ? gly[c] + ? ala_L[c] + ? val_L[c] + ? leu_L[c] + ? ile_L[c] + ? met_L[c] + ? trp_L[c] + ? phe_L[c] + ? pro_L[c] + ? ser_L[c] + ? thr_L[c] + ? cys_L[c] + ? tyr_L[c] + ? asn_L[c] + ? gln_L[c] + ? glu_L[c] + ? asp_L[c] + ? lys_L[c] + ? arg_L[c] + ? his_L[c] -> ? h[c] + ? amp[c] + adp[c] + ? pi[c] + ? gdp[c] + ? ppi[c] + XXX[c]"
    translationFormula = substitute_AAs_count(templateFormula, AAcounts)
    translationFormula = translationFormula.replace("? h2o[c]",str(2*N-1)+" h2o[c]")
    translationFormula = translationFormula.replace("? h[c]",str(2*N-1)+" h[c]")
    translationFormula = translationFormula.replace("? ppi[c]",str(N)+" ppi[c]")
    translationFormula = translationFormula.replace("? pi[c]",str(2*N-1)+" pi[c]")
    translationFormula = translationFormula.replace("? gdp[c]",str(2*N-2)+" gdp[c]")
    translationFormula = translationFormula.replace("? amp[c]",str(N)+" amp[c]")
    translationFormula = translationFormula.replace("? gtp[c]",str(2*N-2)+" gtp[c]")
    translationFormula = translationFormula.replace("? atp[c]",str(N+1)+" atp[c]")
    translationFormula = translationFormula.replace("XXX[c]",str(entryID)+"[c]")
    return translationFormula

# Function that replaces the XXX template name for the UniProt ID
def insert_prot_name_in_rxnFormula(formula,entryID):
    newFormula = formula.replace("XXX",str(entryID))
    return newFormula

# Function that replaces the XXX template name for the Reaction abbreviation
def insert_prot_name_in_rxnName(rxnAbbrev,entryID):
    newAbbreviation = str(entryID) + "_" + rxnAbbrev
    return newAbbreviation

#Function for adding the reactions of a given PathwayName
def addPathway(pathwayName,listOfRxns,listOfRxnsNames):   
    newList = listOfRxns    
    newList2 = listOfRxnsNames
    for i in range(len(rxnPathway)):
        if rxnPathway[i] == pathwayName:
            newList.append(rxnFormula[i])
            newList2.append(rxnAbbreviation[i])
    return newList,newList2

#Function for adding the reactions of a given PathwayName given a condition
def addPathwayFromCondition(conditionName,listOfRxns,listOfRxnsNames):   
    newList = listOfRxns 
    newList2 = listOfRxnsNames
    for i in range(len(rxnConditions)):
        if rxnConditions[i] == conditionName:
            newList.append(rxnFormula[i])
            newList2.append(rxnAbbreviation[i])
    return newList,newList2

#Function for creating a list of GPRs given a list of Rxn names
def getGPRsFromRxnNames(listOfRxnsNames):
    GPR_list = []
    for reaction in listOfRxnsNames:
        if reaction in rxnAbbreviation:
            GPR_list.append(rxnGPR[rxnAbbreviation.index(reaction)])
        else:
            GPR_list.append('')
    return GPR_list
     
#Function that adds canonical reactions to overall list
def addCanonicalRxns(listOfRxns,listOfRxnNames,listOfGPRs):
    newListRxns = listOfRxns
    newListNames = listOfRxnNames
    newListGPRs = listOfGPRs
    #Add  canonical reactions
    [newListRxns, newListNames] = addPathway("Canonical",listOfRxns,listOfRxnNames)
    #Add canonical GPRs
    r = []
    n = []
    [r,n] = addPathway("Canonical",r,n)
    newGPRs = getGPRsFromRxnNames(n)
    for gpr in newGPRs:
        newListGPRs.append(gpr)
    return newListRxns, newListNames, newListGPRs

In [25]:
import pandas as pd
d = pd.read_csv(f_path + 'PSIM_HUMAN.tab',sep = '\t')
d.iloc[2,[0,4,5,6,7,8,9,10]]

Entry       P06865
SP               1
DSB              3
GPI              0
NG               3
OG               0
TMD              0
Location       [l]
Name: 2, dtype: object

In [544]:
entryID = 'P06865'
PSI_row = PSIM[PSIM_entries.index(str(entryID))] 
PSI_row = PSI_row.split('\t')
sequence = PSI_row[11]
L = float(PSI_row[2]) # Protein Length
MW = float(PSI_row[3]) # Molecular weight
PSI = []
Kv = 0.7
V = MW * 1.21 / 1000.0 # Protein Volume in nm^3
clathrin_coeff = int(round(29880.01 * Kv / V)) # Number of proteins per clathrin vesicle  
copi_coeff = int(round(143793.19 * Kv / V))
copii_coeff = int(round(268082.35 * Kv / V))
connector = ''
#Prepare vectors that will store reactions and components
protName = str(entryID)
rxns = []
rxnNames = []   

#Add translation reaction
translation_reaction = translate_protein(protName)
rxns.append(translation_reaction)
rxnNames.append("TRANSLATION_protein") 
for i in [0,4,5,6,7,8,9,10]:
    PSI.append(PSI_row[i])  #This is the vector from the PSIM that corresponds to the given protein 
                                      #[entry,SP,DSB,GPI,NG,OG,TMD,SubCellLoc] 
PSI[7] = 'e'
if PSI[1] == '0': #If it doesn't have signal peptide then ignore
    raise ValueError('NAAWWWW')

elif PSI[1] == '1': #Translocate protein if it has signal peptide
    if L <= 160:
        [rxns,rxnNames] = addPathway("Post-translational Translocation",rxns,rxnNames)
        if PSI[7] == '[e]' or PSI[7] == '':
            [rxns,rxnNames] = addPathway("Post-translational Translocation (Secretory protein)",rxns,rxnNames)
        if PSI[7] == "[pm]" or PSI[6] != '0':
            [rxns,rxnNames] = addPathway("Post-translational Translocation (Tail anchored membrane protein)",rxns,rxnNames)
    else:
        [rxns,rxnNames] = addPathway("Translocation",rxns,rxnNames)

    number_BiP = L/40 #Number of BiPs depends on protein length http://www.cshperspectives.com/content/5/5/a013201.full
    for i in range(len(rxns)):
        rxns[i] = rxns[i].replace("!",str(number_BiP))
    connector = 'XXX[r]'

In [545]:


#----------------
# number_DSB = '2'
# DSBrxns = []
# DSBrxnNames = []
# DSBrxns.append(connector + ' -> XXX_preDSB[r]')
# DSBrxnNames.append('Start_DSB')
# [DSBrxns, DSBrxnNames] = addPathwayFromCondition('DSB>0',DSBrxns,DSBrxnNames)
# for i in range(len(DSBrxns)):
#     if '?' in DSBrxns[i]:
#         if number_DSB == '1':
#             DSBrxns[i] = DSBrxns[i].replace('? ','')
#         else:
#             DSBrxns[i] = DSBrxns[i].replace('?',number_DSB)

# connector = 'XXX_DSB[r]'
# for reaction in DSBrxns:
#         rxns.append(reaction)
# for reactionName in DSBrxnNames:
#         rxnNames.append(reactionName)
# [rxns,rxnNames] = addPathway('COPII_DSB',rxns,rxnNames)
# copii_rxns = []
# copii_names = []
# [copii_rxns, copii_names] =  addPathway('COPII-canonical',copii_rxns,copii_names)       
# for i in range(len(copii_rxns)):
#     copii_rxns[i] = copii_rxns[i].replace("!",str(copii_coeff))
#     rxns.append(copii_rxns[i])
#     rxnNames.append(copii_names[i])
# dsb_rxns = rxns.copy()

#---------
# rxns.append(connector + ' -> XXX_preGPI[r]')
# rxnNames.append('Start_GPI')
# [rxns,rxnNames] = addPathwayFromCondition('GPI=1',rxns,rxnNames)
# connector = 'XXX-dgpi_hs[r]'
# copii_rxns = []
# copii_names = []
# [copii_rxns, copii_names] =  addPathway('COPII_GPI',copii_rxns,copii_names)       
# for i in range(len(copii_rxns)):
#     copii_rxns[i] = copii_rxns[i].replace("!",str(copii_coeff))
#     rxns.append(copii_rxns[i])
#     rxnNames.append(copii_names[i])
# gpi_reactions = rxns.copy()


#--------n glycosylation er
rxns.append(connector + ' -> XXX_preNG[r]')
rxnNames.append('Start_NG')
number_Nglycans = '3' #Get number of N-Glycans
NGlyrxns = []
NGlyrxnNames = []

[NGlyrxns,NGlyrxnNames] = addPathwayFromCondition('NG>0',NGlyrxns,NGlyrxnNames)
[NGlyrxns,NGlyrxnNames] = addPathway('Golgi processing N',NGlyrxns,NGlyrxnNames)
# for i in range(len(NGlyrxns)): #Change the '?' for the number of N-glycans
#     if NGlyrxns[i] == 'XXX-M5-unfold-UBIQP[c] + ? h2o[c] + RAD23A[c] =>  XXX-unfold-UBIQP-RAD23A[c] + ? acgam[c] + ? man[c]':
#         h2o = str(7*int(number_Nglycans))
#         acgam = str(2*int(number_Nglycans))
#         man = str(5*int(number_Nglycans))
#         NGlyrxns[i] = 'XXX-M5-unfold-UBIQP[c] + ' + h2o +' h2o[c] + RAD23A[c] =>  XXX-unfold-UBIQP[c]-RAD23A[c] + ' + acgam + ' acgam[c] + ' + man +' man[c]'
#     if '?' in NGlyrxns[i]:
#             if number_Nglycans == '1':
#                 NGlyrxns[i] = NGlyrxns[i].replace('? ', '')
#             else:
#                 NGlyrxns[i] = NGlyrxns[i].replace('?', number_Nglycans)

for reaction in NGlyrxns:
    rxns.append(reaction)
for reactionName in NGlyrxnNames:
    rxnNames.append(reactionName)
copii_rxns = []
copii_names = []
[copii_rxns, copii_names] =  addPathway('COPII_NG',copii_rxns,copii_names)       
for i in range(len(copii_rxns)):
    copii_rxns[i] = copii_rxns[i].replace("!",str(copii_coeff))
    rxns.append(copii_rxns[i])
    rxnNames.append(copii_names[i])
n_reactions = rxns.copy()

# # O glycosylation---------------------------------------------------------
# rxns.append(connector + ' -> XXX_preOG[g]')
# rxnNames.append('Start_OG')
# number_Oglycans = '3' #Get number of O-Glycans
# OGlyrxns = []
# OGlyrxnNames = []

# [OGlyrxns,OGlyrxnNames] = addPathwayFromCondition('OG>0',OGlyrxns,OGlyrxnNames)
# for i in range(len(OGlyrxns)): #Change the '?' for the number of O-glycans
#     if '?' in OGlyrxns[i]:
#             if number_Oglycans == '1':
#                 OGlyrxns[i] = OGlyrxns[i].replace('? ', '')
#             else:
#                 OGlyrxns[i] = OGlyrxns[i].replace('?', number_Oglycans)

# for reaction in OGlyrxns:
#     rxns.append(reaction)
# for reactionName in OGlyrxnNames:
#     rxnNames.append(reactionName)

# connector = 'XXX-Core2[g]'

# Golgi reactions---------------------------------------------------------
# if 'SP_degradation' in rxnNames:
#         SPaas = count_AAs(sequence[0:22]) #Amino acids in signal peptide assuming length is 22 on average
#         rxns[rxnNames.index('SP_degradation')] = substitute_AAs_count(rxns[rxnNames.index('SP_degradation')],SPaas)
#     if 'Ubiquitination_degradation' in rxnNames:
#         new_aas = count_AAs(sequence[22:])
#         rxns[rxnNames.index('Ubiquitination_degradation')] = rxns[rxnNames.index('Ubiquitination_degradation')].replace("!", "?")
#         rxns[rxnNames.index('Ubiquitination_degradation')] = substitute_AAs_count(rxns[rxnNames.index('Ubiquitination_degradation')], new_aas)
#         rxns[rxnNames.index('Ubiquitination_degradation')] = rxns[rxnNames.index('Ubiquitination_degradation')].replace("?",str(L-22)) 


# COPI-----------------------------------------------
# rxns.append(connector + ' -> XXX_preCOPI[g]')
# rxnNames.append('Start_COPI')

# copi_rxns = []
# copi_names = []
# [copi_rxns, copi_names] =  addPathway('COPI',copi_rxns,copi_names)       
# for i in range(len(copi_rxns)):
#     copi_rxns[i] = copi_rxns[i].replace("!",str(copi_coeff))
#     rxns.append(copi_rxns[i])
#     rxnNames.append(copi_names[i])

# connector = 'XXX_mature[r]'
# location = PSI[7]
# if location == '[r]':
#     rxns.append(connector + ' -> ')
#     rxnNames.append(protName + '_Final_demand')
# elif location == '[rm]':
#     rxns.append(connector + ' -> XXX_mature' + location)
#     rxnNames.append('Final_location_' + location)
#     rxns.append('XXX_mature' + location + ' -> ')
#     rxnNames.append(protName + '_Final_demand')


# # clathrin transport to lysosome-----------------------------------------------
# rxns.append(connector + ' -> XXX-preClathrin[g]')
# rxnNames.append('Start_Clathrin_vesicle')

# clath_rxns = []
# clath_names = []
# [clath_rxns, clath_names] =  addPathway('Clathrin vesicles',clath_rxns,clath_names)       
# for i in range(len(clath_rxns)):
#     clath_rxns[i] = clath_rxns[i].replace("!",str(clathrin_coeff))
#     rxns.append(clath_rxns[i])
#     rxnNames.append(clath_names[i])        

# connector = 'XXX_mature[cv]'
# location = 'l'
# rxns.append(connector + ' -> XXX_mature' + location)
# rxnNames.append('Final_location_' + location)
# rxns.append('XXX_mature' + location + ' -> ')
# rxnNames.append(protName + '_Final_demand')

In [ ]:
[rxns,rxnNames] = addPathway('COPII-normal',rxns,rxnNames)
        
copii_rxns = []
copii_names = []
[copii_rxns, copii_names] =  addPathway('COPII-canonical',copii_rxns,copii_names)       
for i in range(len(copii_rxns)):
    copii_rxns[i] = copii_rxns[i].replace("!",str(copii_coeff))
    rxns.append(copii_rxns[i])
    rxnNames.append(copii_names[i])        

#[rxns,rxnNames] = addPathway('COPII-canonical',rxns,rxnNames)
connector = 'XXX[g]'

In [530]:
len(rxns)

43

In [519]:
rxns.append(connector + ' -> XXX-preSV[g]')
rxnNames.append('Start_Secretion')

sv_rxns = []
sv_names = []
[sv_rxns, sv_names] =  addPathway('SV',sv_rxns, sv_names)       
for i in range(len(sv_rxns)):
    sv_rxns[i] = sv_rxns[i].replace("!",str(clathrin_coeff))
    rxns.append(sv_rxns[i])
    rxnNames.append(sv_names[i])        

#[rxns,rxnNames] = addPathway("SV",rxns,rxnNames)
if location == '':
    location = '[e]'
rxns.append('XXX_mature[sv]' + ' -> XXX_mature' + location)
rxnNames.append('Final_location_' + location)
rxns.append('XXX_mature' + location + ' -> ')
rxnNames.append('Final_demand')

In [522]:
clathrin_coeff

285

In [546]:
rxns[13:]

['XXX-M7A-misfold[r] + ? h2o[r] -> XXX-M6-misfold[r] + ? man[r]',
 'XXX-M6-misfold[r] + ? h2o[r] -> XXX-M5-misfold[r] + ? man[r]',
 'XXX-M5-misfold[r] + OS9[r] + BiP-adp[r] + HSP90B1[r] -> XXX-M5-unfold-OS9-BiP-adp-HSP90B1[r]',
 'XXX-M5-unfold-OS9-SEL1[r] + SYVN1[r] -> XXX-M5-unfold-SEL1-SYVN1[r] + OS9[r]',
 'XXX-M5-unfold-SEL1-SYVN1[r] + UBIQP[c] + 8 atp[c] -> XXX-M5-unfold-UBIQP-SEL1-SYVN1[r] + 8 amp[c] + 8 ppi[c]',
 'XXX-M5-unfold-UBIQP-SEL1-SYVN1[r] + retroTranslocase[c] + 6 atp[c] + 6 h2o[c] -> XXX-M5-unfold-UBIQP[c] + SYVN1[r] + 6 adp[c] + 6 pi[c] + SEL1[r] + NPLOC4[c] + UFD1L[c] + VCP[c] + DERL1[r] + DERL3[r] + VIMP[c] + BCAP31[r] + SEC61C[r]',
 'XXX-M5-unfold-UBIQP[c] + ? h2o[c] + RAD23A[c] ->  XXX-unfold-UBIQP-RAD23A[c] + ? acgam[c] + ? man[c]',
 'XXX-unfold-UBIQP-RAD23A[c] + Proteasome[c] + 8 h2o[c] ->  XXX-unfold-Proteasome[c] + RAD23A[c] + UBIQP[c]',
 'XXX-unfold-Proteasome[c] + ! atp[c] + ! h2o[c] -> ! gly[c] + ! ala_L[c] + ! val_L[c] + ! leu_L[c] + ! ile_L[c] + ! met_L[c]

In [547]:
rxnNames[18]

'retro_TRANSLOC_2'

In [548]:
rxns[18]

'XXX-M5-unfold-UBIQP-SEL1-SYVN1[r] + retroTranslocase[c] + 6 atp[c] + 6 h2o[c] -> XXX-M5-unfold-UBIQP[c] + SYVN1[r] + 6 adp[c] + 6 pi[c] + SEL1[r] + NPLOC4[c] + UFD1L[c] + VCP[c] + DERL1[r] + DERL3[r] + VIMP[c] + BCAP31[r] + SEC61C[r]'

In [549]:
GPRs = getGPRsFromRxnNames(rxnNames)
for i in range(len(rxns)):
    rxns[i] = insert_prot_name_in_rxnFormula(rxns[i],protName)
for i in range(len(rxnNames)):
    rxnNames[i] = insert_prot_name_in_rxnName(rxnNames[i],protName)

In [550]:
GPRs[18]

'(6400) and (84447) and (55666) and (7353) and (7415) and (79139) and (55829) and (91319) and (10134)'

In [551]:
len(GPRs)

43

In [553]:
rxns[18]

'P06865-M5-unfold-UBIQP-SEL1-SYVN1[r] + retroTranslocase[c] + 6 atp[c] + 6 h2o[c] -> P06865-M5-unfold-UBIQP[c] + SYVN1[r] + 6 adp[c] + 6 pi[c] + SEL1[r] + NPLOC4[c] + UFD1L[c] + VCP[c] + DERL1[r] + DERL3[r] + VIMP[c] + BCAP31[r] + SEC61C[r]'